### 1 задание
Напишите функцию create_model, которая должна возвращать полносвязную нейронную сеть из двух слоев. 
На вход должно быть 100 чисел, на выход 1, посередине 10. В качестве нелинейности используйте ReLU. 
Воспользуйтесь nn.Sequential и передайте слои как последовательность.

In [16]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import Optimizer
from torchvision.datasets import MNIST
from tqdm import tqdm



In [17]:
def create_model(n_features=100, middle=10, out_features=1):
    ### задаем сеть через последовательность
    net = nn.Sequential(
        nn.Linear(n_features, middle), ### первый слой 
        nn.ReLU(), ### функция активации
        nn.Linear(middle, out_features) ### второй слой
    )
    return net

### 2 задание
Напишите функцию train. Она должна принимать на вход нейронную сеть, даталоадер, оптимизатор и функцию потерь. Она должна иметь следующую сигнатуру: def train(model: nn.Module, data_loader: DataLoader, optimizer: Optimizer, loss_fn):

Внутри функции сделайте следующие шаги:

1. Переведите модель в режим обучения.

2. Проитерируйтесь по даталоадеру.

3. На каждой итерации:

    - Занулите градиенты с помощью оптимизатора

    - Сделайте проход вперед (forward pass)

    - Посчитайте ошибку

    - Сделайте проход назад (backward pass)

    - Напечатайте ошибку на текущем батче с точностью до 5 символов после запятой (только число)

    - Сделайте шаг оптимизации

Функция должна вернуть среднюю ошибку за время прохода по даталоадеру.

In [18]:
def train(model: nn.Module, data_loader: DataLoader, optimizer: Optimizer, loss_fn):
    model.train() ### перевод в режим обучения
    loss_1 = 0 ### задаем начало подсчета лосса
    for x,y in tqdm(data_loader): ### итерируемся по даталоадеру, можно было и без tqdm 
        optimizer.zero_grad() ### зануляем градиенты
        output = model(x) ### считаем выходы
        loss = loss_fn(output, y) ### считаем ошибку
        loss_1 += loss.item() ### добавляем ошибку
        loss.backward() ### делаем обратный проход, считаем градиенты
        print(f"{loss.item():.5f}") ### печатаем текущую ошибку
        optimizer.step() ### делаем шаг оптимизатора, обновляя веса модели..
    
    mean = loss_1/(len(data_loader)) ### усредняем ошибку
    return mean   ### возвращаем усредненную ошибку


### 3 задание
Напишите функцию evaluate. Она должна принимать на вход нейронную сеть, даталоадер и функцию потерь. Она должна иметь следующую сигнатуру: def evaluate(model: nn.Module, data_loader: DataLoader, loss_fn):

Внутри функции сделайте следующие шаги:

1. Переведите модель в режим инференса (применения)

2. Проитерируйтесь по даталоадеру

3. На каждой итерации:

    - Сделайте проход вперед (forward pass)

    - Посчитайте ошибку

Функция должна вернуть среднюю ошибку за время прохода по даталоадеру.

In [19]:
@torch.inference_mode() ### навешиваем декоратор для запрета подсчета градиентов
def evaluate(model:nn.Module, data_loader:DataLoader, loss_fn):
    model.eval() ### на всякий случай включаем модель в режим eval
    loss_all = 0 ### начало подсчета ошибки
    for x,y in tqdm(data_loader): ### итерируемся
        output = model(x) ### форврад пасс
        loss = loss_fn(output, y) ### считаем ошибку
        loss_all +=loss.item() ### добавляем ошибку
    mean_loss = loss_all/(len(data_loader)) ### считаем среднее
    return mean_loss ### возвращаем среднюю ошибку